Extract data from Covid-19 csv file and store them in database

Load Data

In [5]:
import pandas as pd

# Load the CSV file into a pandas DataFrame
df = pd.read_csv('../CSV Files/covid.csv')

# Display the first few rows of the DataFrame
print("Loaded data:")
print(df.head())

# Check for missing values
missing_values = df.isnull().sum()
print("\nMissing values in each column:")
print(missing_values)

# Display data types of columns
data_types = df.dtypes
print("\nData types of columns:")
print(data_types)


Loaded data:
  Country/Region  Confirmed  Deaths  Recovered  Active  New cases  New deaths  \
0    Afghanistan      36263    1269      25198    9796        106          10   
1        Albania       4880     144       2745    1991        117           6   
2        Algeria      27973    1163      18837    7973        616           8   
3        Andorra        907      52        803      52         10           0   
4         Angola        950      41        242     667         18           1   

   New recovered  Deaths / 100 Cases  Recovered / 100 Cases  \
0             18                3.50                  69.49   
1             63                2.95                  56.25   
2            749                4.16                  67.34   
3              0                5.73                  88.53   
4              0                4.32                  25.47   

   Deaths / 100 Recovered  Confirmed last week  1 week change  \
0                    5.04                35526          

Clean Data

In [6]:
# Adjust column names to match the database schema
df.columns = [
    'country_region', 'confirmed', 'deaths', 'recovered', 'active', 
    'new_cases', 'new_deaths', 'new_recovered', 'deaths_per_100_cases', 
    'recovered_per_100_cases', 'deaths_per_100_recovered', 
    'confirmed_last_week', 'one_week_change', 'one_week_percent_increase', 
    'who_region'
]

# Optionally handle missing values (e.g., fill with 0 or drop rows)
df.fillna(0, inplace=True)
print("\nData after handling missing values:")
print(df.head())



Data after handling missing values:
  country_region  confirmed  deaths  recovered  active  new_cases  new_deaths  \
0    Afghanistan      36263    1269      25198    9796        106          10   
1        Albania       4880     144       2745    1991        117           6   
2        Algeria      27973    1163      18837    7973        616           8   
3        Andorra        907      52        803      52         10           0   
4         Angola        950      41        242     667         18           1   

   new_recovered  deaths_per_100_cases  recovered_per_100_cases  \
0             18                  3.50                    69.49   
1             63                  2.95                    56.25   
2            749                  4.16                    67.34   
3              0                  5.73                    88.53   
4              0                  4.32                    25.47   

   deaths_per_100_recovered  confirmed_last_week  one_week_change  \
0   

Store the data in database

In [7]:
import psycopg2
from psycopg2 import sql

# Connect to database
conn = psycopg2.connect(
    dbname="covid",
    user="postgres",
    password="KARU55bime22", 
    host="localhost", 
    port="5432"      
)

# Create a cursor object
cursor = conn.cursor()

# Create the covid_data table
create_table_query = """
CREATE TABLE IF NOT EXISTS covid_data (
    country_region VARCHAR(100),
    confirmed BIGINT,
    deaths BIGINT,
    recovered BIGINT,
    active BIGINT,
    new_cases BIGINT,
    new_deaths BIGINT,
    new_recovered BIGINT,
    deaths_per_100_cases FLOAT,
    recovered_per_100_cases FLOAT,
    deaths_per_100_recovered FLOAT,
    confirmed_last_week BIGINT,
    one_week_change BIGINT,
    one_week_percent_increase FLOAT,
    who_region VARCHAR(50)
);
"""
cursor.execute(create_table_query)
conn.commit()

# Insert the data
for index, row in df.iterrows():
    insert_query = """
    INSERT INTO covid_data (
        country_region, confirmed, deaths, recovered, active, 
        new_cases, new_deaths, new_recovered, deaths_per_100_cases, 
        recovered_per_100_cases, deaths_per_100_recovered, 
        confirmed_last_week, one_week_change, one_week_percent_increase, 
        who_region
    ) VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
    """
    cursor.execute(insert_query, tuple(row))

conn.commit()

# Query the database to check the contents
select_query = "SELECT * FROM covid_data LIMIT 5;"
cursor.execute(select_query)
result = cursor.fetchall()

# Close the connection
cursor.close()
conn.close()
print("Data successfully inserted into the database.")


Data successfully inserted into the database.


 Extract the data from weather data CSV file and store them in database

Load Data

In [9]:
import pandas as pd


# Load the weather CSV file into a pandas DataFrame
df = pd.read_csv('../CSV Files/weather.csv')

# Display the first few rows
print("Loaded data:")
print(df.head())

# Check for missing values
missing_values = df.isnull().sum()
print("\nMissing values in each column:")
print(missing_values)

# Display data types
data_types = df.dtypes
print("\nData types of columns:")
print(data_types)


Loaded data:
                  Formatted Date        Summary Precip Type  Temperature (C)  \
0  2006-04-01 00:00:00.000 +0200  Partly Cloudy        rain         9.472222   
1  2006-04-01 01:00:00.000 +0200  Partly Cloudy        rain         9.355556   
2  2006-04-01 02:00:00.000 +0200  Mostly Cloudy        rain         9.377778   
3  2006-04-01 03:00:00.000 +0200  Partly Cloudy        rain         8.288889   
4  2006-04-01 04:00:00.000 +0200  Mostly Cloudy        rain         8.755556   

   Apparent Temperature (C)  Humidity  Wind Speed (km/h)  \
0                  7.388889      0.89            14.1197   
1                  7.227778      0.86            14.2646   
2                  9.377778      0.89             3.9284   
3                  5.944444      0.83            14.1036   
4                  6.977778      0.83            11.0446   

   Wind Bearing (degrees)  Visibility (km)  Loud Cover  Pressure (millibars)  \
0                   251.0          15.8263         0.0           

Clean the Data

In [ ]:
# Drop the 'Precip Type' column(missing value)
df.drop(columns=['Precip Type'], inplace=True)

# Verify the column has been dropped
print("\nRemaining columns after dropping 'Precip Type':")
print(df.columns)

# Convert the 'Formatted Date' column to datetime format
df['Formatted Date'] = pd.to_datetime(df['Formatted Date'])

# Rename columns to match the database schema
df.columns = [
    'Formatted_Date', 'Summary', 'Temperature_C', 
    'Apparent_Temperature_C', 'Humidity', 'Wind_Speed_kmph', 
    'Wind_Bearing_degrees', 'Visibility_km', 'Cloud_Cover', 
    'Pressure_millibars', 'Daily_Summary'
]

# Display the cleaned data
print("\nCleaned data preview:")
print(df.head())



Remaining columns after dropping 'Precip Type':
Index(['Formatted Date', 'Summary', 'Temperature (C)',
       'Apparent Temperature (C)', 'Humidity', 'Wind Speed (km/h)',
       'Wind Bearing (degrees)', 'Visibility (km)', 'Loud Cover',
       'Pressure (millibars)', 'Daily Summary'],
      dtype='object')

Cleaned data preview:
              Formatted_Date        Summary  Temperature_C  \
0  2006-04-01 00:00:00+02:00  Partly Cloudy       9.472222   
1  2006-04-01 01:00:00+02:00  Partly Cloudy       9.355556   
2  2006-04-01 02:00:00+02:00  Mostly Cloudy       9.377778   
3  2006-04-01 03:00:00+02:00  Partly Cloudy       8.288889   
4  2006-04-01 04:00:00+02:00  Mostly Cloudy       8.755556   

   Apparent_Temperature_C  Humidity  Wind_Speed_kmph  Wind_Bearing_degrees  \
0                7.388889      0.89          14.1197                 251.0   
1                7.227778      0.86          14.2646                 259.0   
2                9.377778      0.89           3.9284         

C:\Users\win10\AppData\Local\Temp\ipykernel_5228\1929571021.py:9: FutureWarning: In a future version of pandas, parsing datetimes with mixed time zones will raise an error unless `utc=True`. Please specify `utc=True` to opt in to the new behaviour and silence this warning. To create a `Series` with mixed offsets and `object` dtype, please use `apply` and `datetime.datetime.strptime`
  df['Formatted Date'] = pd.to_datetime(df['Formatted Date'])


In [13]:
import psycopg2
from psycopg2 import sql

# Connect to database
conn = psycopg2.connect(
    dbname="weather",
    user="postgres",  
    password="KARU55bime22",  
    host="localhost",  
    port="5432" 
)

cursor = conn.cursor()

# Create the weather_data table
create_table_query = """
CREATE TABLE IF NOT EXISTS weather_data (
    Formatted_Date DATE,
    Summary TEXT,
    Temperature_C FLOAT,
    Apparent_Temperature_C FLOAT,
    Humidity FLOAT,
    Wind_Speed_kmph FLOAT,
    Wind_Bearing_degrees INT,
    Visibility_km FLOAT,
    Cloud_Cover FLOAT,
    Pressure_millibars FLOAT,
    Daily_Summary TEXT
);
"""
cursor.execute(create_table_query)
conn.commit()

# Insert data
for index, row in df.iterrows():
    insert_query = """
    INSERT INTO weather_data (
        Formatted_Date, Summary, Temperature_C, 
        Apparent_Temperature_C, Humidity, Wind_Speed_kmph, 
        Wind_Bearing_degrees, Visibility_km, Cloud_Cover, 
        Pressure_millibars, Daily_Summary
    ) VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
    """
    cursor.execute(insert_query, tuple(row))

conn.commit()

select_query = "SELECT * FROM weather_data LIMIT 5;"
cursor.execute(select_query)
result = cursor.fetchall()

cursor.close()
conn.close()
print("Data successfully inserted into the database.")



Data successfully inserted into the database.
